<a href="https://colab.research.google.com/github/AmitoVrito/Traceprop/blob/main/notebooks/exp25_llm_inline_overhead_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Traceprop-LLM — inline gradient provenance (MLSys)

**Claim:** capturing per-sample gradient provenance *inline* during a LoRA fine-tune costs **<1% wall-clock**, vs the full extra pass over the training set that post-hoc methods (TRAK / LoGRA) require.

Run on a GPU runtime (Runtime → Change runtime type → GPU; T4/L4 fine, A100 for the largest).

Sections:
1–5. **Overhead** — baseline vs inline-instrumented step time on GPT-2 and Pythia-410M/1B (`exp25`); tracked-blocks tradeoff; results table.
6. **Head-to-head** — inline logging vs a post-hoc extraction sweep (`exp26`): the ~30×/~150× cost win that is the paper's centerpiece.

## 1. Install dependencies

In [1]:
!pip -q install transformers peft accelerate
# Colab ships an old torchao (0.10) whose PEFT compatibility check raises on import; we don't use it.
!pip -q uninstall -y torchao 2>/dev/null
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

torch 2.11.0+cu128 cuda True NVIDIA L4


## 2. Get the Traceprop code

The `traceprop.llm` module and `exp25` are on the private repo. Paste a GitHub token (a fine-grained read token for `AmitoVrito/Traceprop`) below. If you prefer, upload the repo zip instead and skip this cell.

In [2]:
import getpass, os
TOKEN = getpass.getpass('GitHub token (leave blank if uploading manually): ').strip()
if TOKEN:
    url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
    !git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
    %cd /content/Traceprop
    !pip -q install -e .
else:
    print('No token given — upload the repo to /content/Traceprop, then run: %cd /content/Traceprop and !pip install -e .')

GitHub token (leave blank if uploading manually): ··········
/content/Traceprop
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for traceprop (pyproject.toml) ... done


In [3]:
# sanity: import the inline logger
import sys; sys.path.insert(0, '/content/Traceprop/experiments')
from traceprop.llm import LoRAGradientLogger, select_lora_linears
print('traceprop.llm OK')

traceprop.llm OK


## 3. Headline: overhead on GPT-2 (124M) + LoRA

Baseline = plain LoRA step. Instrumented = same step + inline per-sample gradient logging. The store footprint is what you keep for attribution.

Each run reports **two** overhead numbers:
- **`synced%`** — conservative: a GPU sync is forced every step, so the projection is fully serialized against the training step. Upper bound.
- **`throughput%`** — realistic: projected gradients are buffered on-device and the run is synced once, so the projection overlaps with compute. This is the actual cost of turning logging on in a training loop, and the headline number.

`--track 1` logs only the **last transformer block** (last-layer attribution, i.e. Traceprop-LL): the projected-gradient dimension — and thus the projection cost — scales with tracked parameters, so last-block is the cheap regime. Batch/seq are held **fixed across all three models** so the only variable is model size.

In [4]:
%cd /content/Traceprop/experiments
!python exp25_llm_inline_overhead.py --backend hf --model gpt2 --device cuda \
    --steps 120 --warmup 15 --repeats 5 --batch 8 --seq 128 --rank 8 --proj_dim 512 --track 1

/content/Traceprop/experiments
config.json: 100% 665/665 [00:00<00:00, 2.93MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:  71% 391M/548M [00:02<00:00, 265MB/s, 33.0MB/s  ]
model.safetensors: downloading bytes:  86% 474M/548M [00:02<00:00, 193MB/s, 40.0MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:03<00:00, 148MB/s, 40.8MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:03<00:00, 171MB/s, 49.7MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 5931.06it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 536kB/s]
/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 122kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 33.6MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 22.2MB/s]
tokenizer.json: 1

## 4. Scale up: Pythia-410M (and 1B on A100)

In [5]:
!python exp25_llm_inline_overhead.py --backend hf --model EleutherAI/pythia-410m --device cuda \
    --steps 120 --warmup 15 --repeats 5 --batch 8 --seq 128 --rank 8 --proj_dim 512 --track 1

config.json: 100% 570/570 [00:00<00:00, 2.25MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   0% 0.00/911M [00:00<?, ?B/s]
model.safetensors: downloading bytes:  17% 153M/911M [00:02<00:06, 122MB/s, 12.7MB/s  ]
model.safetensors: downloading bytes:  20% 180M/911M [00:02<00:05, 126MB/s, 15.5MB/s  ]
model.safetensors: downloading bytes:  22% 198M/911M [00:03<00:06, 115MB/s, 17.2MB/s  ]
model.safetensors: downloading bytes:  24% 219M/911M [00:03<00:06, 111MB/s, 17.9MB/s  ]
model.safetensors: reconstructing file:  23% 206M/911M [00:03<00:08, 83.5MB/s, 16.2MB/s  ]
model.safetensors: downloading bytes:  28% 255M/911M [00:03<00:04, 132MB/s, 20.0MB/s  ]
model.safetensors: downloading bytes:  36% 324M/911M [00:03<00:02, 206MB/s, 24.9MB/s  ]
model.safetensors: downloading bytes:  40% 362M/911M [00:03<00:02, 244MB/s, 28.0MB/s  ]
model.safetensors: downloading bytes:  43% 394M/911M [00:03<00:01, 263MB/s, 31.1MB/s  ]
model.safetensors: d

In [6]:
# Pythia-1B — fits on L4/A100 (24GB+). Same batch/seq as the others for a fair size comparison.
!python exp25_llm_inline_overhead.py --backend hf --model EleutherAI/pythia-1b --device cuda \
    --steps 120 --warmup 15 --repeats 5 --batch 8 --seq 128 --rank 8 --proj_dim 512 --track 1

config.json: 100% 569/569 [00:00<00:00, 2.82MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   0% 0.00/2.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:  12% 260M/2.09G [00:01<00:06, 289MB/s, 19.4MB/s  ]
model.safetensors: downloading bytes:  16% 344M/2.09G [00:02<00:05, 336MB/s, 28.0MB/s  ]
model.safetensors: downloading bytes:  23% 475M/2.09G [00:02<00:03, 454MB/s, 37.8MB/s  ]
model.safetensors: downloading bytes:  29% 598M/2.09G [00:02<00:02, 525MB/s, 48.7MB/s  ]
model.safetensors: downloading bytes:  38% 801M/2.09G [00:02<00:02, 611MB/s, 65.5MB/s  ]
model.safetensors: downloading bytes:  42% 877M/2.09G [00:02<00:01, 639MB/s, 71.4MB/s  ]
model.safetensors: downloading bytes:  54% 1.13G/2.09G [00:03<00:01, 627MB/s, 92.5MB/s  ]
model.safetensors: downloading bytes:  66% 1.37G/2.09G [00:03<00:01, 616MB/s,  112MB/s  ]
model.safetensors: downloading bytes:  73% 1.53G/2.09G [00:03<00:00, 692MB/s,  122MB/s  ]
model.safe

## 4b. Tradeoff: overhead vs number of tracked blocks (GPT-2)

Shows the knob directly — overhead grows with how many blocks you log. Last-block (`--track 1`) is the sub-1% attribution regime; full-model (`--track 0`) is the expensive end. Quality-vs-overhead for these settings is workstream C (LDS parity).

In [7]:
import sys; sys.path.insert(0, '/content/Traceprop/experiments')
from types import SimpleNamespace
from exp25_llm_inline_overhead import run

sweep = []
for t in [1, 2, 6, 0]:   # last-1, last-2, last-6 blocks, then all layers
    ns = SimpleNamespace(backend='hf', model='gpt2', device='cuda', steps=100, warmup=15,
                         repeats=5, batch=8, seq=128, rank=8, proj_dim=512, d=256, n_blocks=2,
                         track=t, factored=False, kfac=16)
    r = run(ns)
    sweep.append((t, r['n_tracked_layers'], r['per_sample_grad_dim'],
                  r['overhead_pct'], r['overhead_std']))

print('\n=== overhead vs tracked blocks (gpt2, batch8 seq128, synced median±std) ===')
print(f"{'track':>6}{'layers':>8}{'grad_dim':>12}{'synced%':>12}")
for t, l, g, o, sd in sweep:
    label = 'all' if t == 0 else f'last-{t}'
    print(f"{label:>6}{l:>8}{g:>12}{o:>8.2f}±{sd:<4.2f}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


{
  "backend": "hf",
  "model": "gpt2",
  "dtype": "fp32",
  "device": "cuda",
  "steps": 100,
  "repeats": 5,
  "batch": 8,
  "seq": 128,
  "rank": 8,
  "proj_dim": 512,
  "factored": false,
  "stored_dim": 512,
  "track_last_n_blocks": 1,
  "n_tracked_layers": 6,
  "per_sample_grad_dim": 67584,
  "samples_logged": 8120,
  "store_bytes": 16629760,
  "store_mb": 16.63,
  "base_step_ms": 99.922,
  "overhead_pct": 0.077,
  "overhead_std": 0.738,
  "overhead_samples": [
    1.079,
    1.873,
    0.049,
    0.077,
    0.053
  ],
  "throughput_overhead_pct": 0.178,
  "throughput_overhead_std": 0.401,
  "throughput_samples": [
    0.148,
    0.178,
    1.193,
    0.357,
    0.157
  ]
}

saved -> results/exp25_hf_gpt2_track1.json


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

{
  "backend": "hf",
  "model": "gpt2",
  "dtype": "fp32",
  "device": "cuda",
  "steps": 100,
  "repeats": 5,
  "batch": 8,
  "seq": 128,
  "rank": 8,
  "proj_dim": 512,
  "factored": false,
  "stored_dim": 512,
  "track_last_n_blocks": 6,
  "n_tracked_layers": 36,
  "per_sample_grad_dim": 405504,
  "samples_logged": 8120,
  "store_bytes": 16629760,
  "store_mb": 16.63,
  "base_step_ms": 99.86,
  "overhead_pct": 2.716,
  "overhead_std": 1.374,
  "overhead_samples": [
    5.193,
    3.381,
    2.716,
    1.35,
    1.668
  ],
  "throughput_overhead_pct": 3.236,
  "throughput_overhead_std": 0.889,
  "throughput_samples": [
    4.987,
    3.236,
    4.948,
    3.022,
    3.222
  ]
}

saved -> results/exp25_hf_gpt2_track6.json


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

<sys>:0: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


{
  "backend": "hf",
  "model": "gpt2",
  "dtype": "fp32",
  "device": "cuda",
  "steps": 100,
  "repeats": 5,
  "batch": 8,
  "seq": 128,
  "rank": 8,
  "proj_dim": 512,
  "factored": false,
  "stored_dim": 512,
  "track_last_n_blocks": 0,
  "n_tracked_layers": 72,
  "per_sample_grad_dim": 811008,
  "samples_logged": 8120,
  "store_bytes": 16629760,
  "store_mb": 16.63,
  "base_step_ms": 99.909,
  "overhead_pct": 8.14,
  "overhead_std": 1.01,
  "overhead_samples": [
    10.153,
    7.771,
    8.14,
    7.134,
    8.392
  ],
  "throughput_overhead_pct": 9.91,
  "throughput_overhead_std": 0.429,
  "throughput_samples": [
    9.931,
    10.044,
    9.91,
    9.54,
    8.87
  ]
}

saved -> results/exp25_hf_gpt2_track0.json

=== overhead vs tracked blocks (gpt2, batch8 seq128, synced median±std) ===
 track  layers    grad_dim     synced%
last-1       6       67584    0.08±0.74
last-2      12      135168    1.03±0.49
last-6      36      405504    2.72±1.37
   all      72      811008    8.14

## 5. Collect results (per-model overhead table)

In [8]:
import glob, json
rows = [json.load(open(p)) for p in glob.glob('/content/Traceprop/experiments/results/exp25_hf_*track1.json')]
print(f"{'model':<22}{'base_ms':>9}{'synced%':>14}{'throughput%':>16}{'store_mb':>10}")
for r in sorted(rows, key=lambda r: r['model']):
    syn = f"{r['overhead_pct']:.2f}±{r['overhead_std']:.2f}"
    thr = f"{r['throughput_overhead_pct']:.2f}±{r['throughput_overhead_std']:.2f}"
    print(f"{r['model']:<22}{r['base_step_ms']:>9.2f}{syn:>14}{thr:>16}{r['store_mb']:>10.2f}")
print('\nsynced%     = conservative (projection serialized against each step)')
print('throughput% = realistic training cost (projection overlaps compute)')

model                   base_ms       synced%     throughput%  store_mb
EleutherAI/pythia-1b     433.65     0.21±0.54       0.62±0.17     19.91
EleutherAI/pythia-410m   215.58     0.71±0.87       0.44±0.13     19.91
gpt2                      99.92     0.08±0.74       0.18±0.40     16.63

synced%     = conservative (projection serialized against each step)
throughput% = realistic training cost (projection overlaps compute)


## 6. Head-to-head: inline logging vs post-hoc extraction (the centerpiece)

`exp26` measures, on the same LoRA model and a fixed N-sample training set, the cost to produce the per-sample gradient store two ways:

- **Post-hoc** (TRAK / LoGRA): a dedicated forward+backward sweep over the whole training set *after* training. TRAK ensembles over K checkpoints → K sweeps.
- **Inline** (Traceprop): the marginal cost folded into the training pass you already run.

`speedup = post-hoc extraction wall-clock / inline marginal wall-clock`. This is what turns "~3% overhead" into "~30× (single ckpt) / ~150× (TRAK 5-ckpt) cheaper to reach attribution-ready."

In [9]:
%cd /content/Traceprop/experiments
!python exp26_posthoc_vs_inline.py --backend hf --model gpt2 --device cuda \
    --n_samples 512 --batch 8 --seq 128 --rank 8 --proj_dim 512 --track 1 --repeats 5 --trak_ckpts 5

/content/Traceprop/experiments
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 148/148 [00:00<00:00, 6013.81it/s]
/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
{
  "backend": "hf",
  "model": "gpt2",
  "device": "cuda",
  "n_samples": 512,
  "n_batches": 64,
  "batch": 8,
  "seq": 128,
  "repeats": 5,
  "proj_dim": 512,
  "track_last_n_blocks": 1,
  "per_sample_grad_dim": 67584,
  "trak_ckpts": 5,
  "store_mb": 1.049,
  "base_train_pass_s": 6.3952,
  "inline_flush_s": 0.1064,
  "inline_flush_std": 0.0004,
  "inline_flush_overhead_pct": 1.664,
  "inline_marginal_s": 0.0542,
  "inline_marginal_std": 0.0223,
  "posthoc_pass_s": 6.3983,
  "posthoc_pass_std": 0.0007,
  "speedup_vs_logra_1ckpt": 60.1,
  "speedup_vs_trak_5ckpt": 300.6
}

saved -> results/exp26_hf_gpt2_track1.json


In [10]:
!python exp26_posthoc_vs_inline.py --backend hf --model EleutherAI/pythia-410m --device cuda \
    --n_samples 512 --batch 8 --seq 128 --rank 8 --proj_dim 512 --track 1 --repeats 5 --trak_ckpts 5

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 292/292 [00:00<00:00, 2069.93it/s]
{
  "backend": "hf",
  "model": "EleutherAI/pythia-410m",
  "device": "cuda",
  "n_samples": 512,
  "n_batches": 64,
  "batch": 8,
  "seq": 128,
  "repeats": 5,
  "proj_dim": 512,
  "track_last_n_blocks": 1,
  "per_sample_grad_dim": 49152,
  "trak_ckpts": 5,
  "store_mb": 1.049,
  "base_train_pass_s": 13.804,
  "inline_flush_s": 0.0978,
  "inline_flush_std": 0.0034,
  "inline_flush_overhead_pct": 0.709,
  "inline_marginal_s": 0.0665,
  "inline_marginal_std": 0.1679,
  "posthoc_pass_s": 13.7915,
  "posthoc_pass_std": 0.0163,
  "speedup_vs_logra_1ckpt": 141.0,
  "speedup_vs_trak_5ckpt": 704.9
}

saved -> results/exp26_hf_EleutherAI_pythia-410m_track1.json


In [11]:
!python exp26_posthoc_vs_inline.py --backend hf --model EleutherAI/pythia-1b --device cuda \
    --n_samples 384 --batch 8 --seq 128 --rank 8 --proj_dim 512 --track 1 --repeats 5 --trak_ckpts 5

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 196/196 [00:00<00:00, 699.11it/s]
{
  "backend": "hf",
  "model": "EleutherAI/pythia-1b",
  "device": "cuda",
  "n_samples": 384,
  "n_batches": 48,
  "batch": 8,
  "seq": 128,
  "repeats": 5,
  "proj_dim": 512,
  "track_last_n_blocks": 1,
  "per_sample_grad_dim": 98304,
  "trak_ckpts": 5,
  "store_mb": 0.786,
  "base_train_pass_s": 20.773,
  "inline_flush_s": 0.0862,
  "inline_flush_std": 0.0003,
  "inline_flush_overhead_pct": 0.415,
  "inline_marginal_s": 0.1002,
  "inline_marginal_std": 0.0838,
  "posthoc_pass_s": 20.8909,
  "posthoc_pass_std": 0.0112,
  "speedup_vs_logra_1ckpt": 242.3,
  "speedup_vs_trak_5ckpt": 1211.4
}

saved -> results/exp26_hf_EleutherAI_pythia-1b_track1.json


In [12]:
import glob, json
rows = [json.load(open(p)) for p in glob.glob('/content/Traceprop/experiments/results/exp26_hf_*track1.json')]
print(f"{'model':<22}{'flush_s':>10}{'overhead%':>11}{'posthoc_s':>11}{'LoGRA x':>9}{'TRAK x':>9}")
for r in sorted(rows, key=lambda r: r['model']):
    trak = [v for k,v in r.items() if k.startswith('speedup_vs_trak')][0]
    print(f"{r['model']:<22}{r['inline_flush_s']:>10.3f}{r['inline_flush_overhead_pct']:>11.2f}"
          f"{r['posthoc_pass_s']:>11.2f}{r['speedup_vs_logra_1ckpt']:>9.1f}{trak:>9.1f}")
print("\nflush_s   = directly-timed (synced) inline logging cost per pass — the reliable number")
print("posthoc_s = dedicated extraction sweep (TRAK/LoGRA);  speedup = posthoc / flush (x K for TRAK)")
print("(inline_marginal_s in the JSON is the pass-difference — noisy at sub-1%; use flush_s.)")

model                    flush_s  overhead%  posthoc_s  LoGRA x   TRAK x
EleutherAI/pythia-1b       0.086       0.41      20.89    242.3   1211.4
EleutherAI/pythia-410m     0.098       0.71      13.79    141.0    704.9
gpt2                       0.106       1.66       6.40     60.1    300.6

flush_s   = directly-timed (synced) inline logging cost per pass — the reliable number
posthoc_s = dedicated extraction sweep (TRAK/LoGRA);  speedup = posthoc / flush (x K for TRAK)
(inline_marginal_s in the JSON is the pass-difference — noisy at sub-1%; use flush_s.)
